The goal of this notebook is to find all three cases of duplicates in the raw TxSON data.

1. **Case one — full_row_duplicates.** 
    - rows that are identical across *every* column.
    - treatment:
        - keep the unique row, drop the duplicates

2. **Case two — full_dup_but_flag**
    - rows that are identical across every column *except* `Flag`.
    - treatment:
        - keeping first occurance
    
3. **Case three - only_duplicate_timestamps**
    - rows that have only an duplicate timestamp
    - treatment:
        - keeping first occurance

We will find these cases, save all the rows of each case in a dataframe, and generate a report on each station.

In [1]:
import pandas as pd
import os

from scripts.read_data import file_to_indexed_df
from scripts.soil_or_met import SoilOrMet

In [2]:
folder = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24' # NOTE: this is the absolute path to 'TxSON_data_2026-02-24' on my local machine

expected_soil_header = "Date,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag" 
expected_met_header = 'TIMESTAMP,RECORD,Rain_mm_Tot,AirTC_Avg,RH,WS_ms_S_WVT,WindDir_D1_WVT,SlrW_Avg,ETos,Rso'

# THE HEADER IS NOT INPUT VALIDATED DUE TO THIS NOT BEING FOR PUBLIC USE, STICK TO ABOVE FORMAT (no space, cap sensitive, no quotes around each column name, etc).

# initialize the soil or met class, which is an object that classifies the raw.dat file based on the above headers and presence of timestamps.
soil_or_met = SoilOrMet(expected_met_header, expected_soil_header, min_soil_features=9, min_met_features=7) # this min_soil_features allows all soil files to be classified

In [3]:
raw_data_dict = {}
soil_counter = 0
met_counter = 0
unknown_counter = 0
for filename in os.listdir(folder):
    
    file_path = folder + '/' + filename

    is_soil_or_met = soil_or_met.determine_data_file(file_path)

    df = file_to_indexed_df(file_path, is_soil_or_met)

    station_name = filename.split('.')[0]

    if df is not None and not df.empty:
        raw_data_dict[station_name] = df

    # counts
    if is_soil_or_met == 'soil':
        soil_counter += 1
    elif is_soil_or_met == 'met':
        met_counter += 1
    elif is_soil_or_met == 'unknown':
        unknown_counter += 1

print(f"Total files processed: {soil_counter + met_counter + unknown_counter}, Soil files: {soil_counter}, Met files: {met_counter}, Unknown files: {unknown_counter}")
print(f"Length of raw_data_dict: {len(raw_data_dict)}, Should be: {soil_counter + met_counter}")

Total files processed: 41, Soil files: 33, Met files: 6, Unknown files: 2
Length of raw_data_dict: 39, Should be: 39


- Now that we have the data in memory, we need to remove two cases of duplicates:
    1. Full row duplicates -- same timestamp, same measurements, same flag (if applicable)
    2. same timestamp, same measurements, different flags

- Then create a dataframe containing all the removed data for that station and by case of duplicate

In [4]:
def treat_case_one_dups(df):
    """
    Assuming timestamps are in the columns. treat case one duplicates.

    Returns a tuple of two dataframes: 
    (1) The dataframe with case one duplicates dropped, keeping the first occurrence.
    (2) The dataframe showing only the case one duplicate rows, keeping all occurrences.
    """
    df_no_case_one_dups = df.drop_duplicates(keep='first')
    df_showing_case_one_dups = df[df.duplicated(keep=False)]
    return df_no_case_one_dups, df_showing_case_one_dups

def treat_case_two_dups(df, ignore_col = 'Flag'):
    """
    Assuming timestamps are in the columns. Treat case two duplicates.
    Returns a tuple of two dataframes: 
    (1) the dataframe with duplicates dropped
    (2) the dataframe showing only the duplicate rows, keeping all occurrences.
    """

    # check if ignore_col is in the dataframe
    if ignore_col not in df.columns:
        print(f"Column '{ignore_col}' not found in the dataframe. No Case two duplicates will be dropped.")
        return df, df.iloc[0:0]  # return the original dataframe and an empty dataframe

    non_target_cols = [col for col in df.columns if col != ignore_col]

    df_no_case_two_dups = df.drop_duplicates(subset=non_target_cols, keep='first')
    df_showing_case_two_dups = df[df.duplicated(subset=non_target_cols, keep=False)]
    
    return df_no_case_two_dups, df_showing_case_two_dups

Lets test out the new tools, first lets track how many dups are found across the stations:

In [5]:
for station, df in raw_data_dict.items():
    print(f"\033[32m\nProcessing {station}...\033[0m") # just prints it in green
    print(f"Original number of rows: {len(df)}")

    df = df.reset_index() # CRUCIAL
    
    df_no_case_one_dups, df_showing_case_one_dups = treat_case_one_dups(df)
    print(f"Number of rows after dropping case one duplicates: {len(df_no_case_one_dups)}")
    
    df_no_case_two_dups, df_showing_case_two_dups = treat_case_two_dups(df_no_case_one_dups, ignore_col='Flag')
    print(f"Number of rows after dropping case two duplicates: {len(df_no_case_two_dups)}")
    
    print("-" * 40)


Processing CB01...
Original number of rows: 85252
Number of rows after dropping case one duplicates: 81185
Number of rows after dropping case two duplicates: 81177
----------------------------------------

Processing CB01_met...
Original number of rows: 15598
Number of rows after dropping case one duplicates: 15597
Column 'Flag' not found in the dataframe. No Case two duplicates will be dropped.
Number of rows after dropping case two duplicates: 15597
----------------------------------------

Processing CB04...
Original number of rows: 93365
Number of rows after dropping case one duplicates: 93358
Number of rows after dropping case two duplicates: 93358
----------------------------------------

Processing CB04_met...
Original number of rows: 99010
Number of rows after dropping case one duplicates: 98811
Column 'Flag' not found in the dataframe. No Case two duplicates will be dropped.
Number of rows after dropping case two duplicates: 98811
----------------------------------------

Pro

lets now look at the df with all the duplicates for a station:

In [6]:
raw_test_df = raw_data_dict['CB01']

# reset index to ensure the index to consider duplicate timestamps
raw_test_df = raw_test_df.reset_index()

# drop case one duplicates
df_no_case_one_dups, df_showing_case_one_dups = treat_case_one_dups(raw_test_df)

print(df_showing_case_one_dups.head(10))


df_unique_dups = df_showing_case_one_dups.drop_duplicates()
print(df_unique_dups.head(5))

                     Date  Ppt  SWC_5  SWC_10  SWC_20  SWC_50    T_5   T_10  \
23935 2017-06-23 00:00:00  0.0  0.040   0.092   0.065   0.065  32.41  33.47   
23936 2017-06-23 00:00:00  0.0  0.040   0.092   0.065   0.065  32.41  33.47   
33086 2018-08-02 11:00:00  0.0  0.049   0.074   0.052   0.054  32.69  30.63   
33087 2018-08-02 11:00:00  0.0  0.049   0.074   0.052   0.054  32.69  30.63   
33088 2018-08-02 12:00:00  0.0  0.052   0.078   0.055   0.057  36.88  33.28   
33089 2018-08-02 12:00:00  0.0  0.052   0.078   0.055   0.057  36.88  33.28   
33090 2018-08-02 12:00:00  0.0  0.052   0.078   0.055   0.057  36.88  33.28   
33091 2018-08-02 13:00:00  0.0  0.053   0.079   0.055   0.057  39.14  34.80   
33092 2018-08-02 13:00:00  0.0  0.053   0.079   0.055   0.057  39.14  34.80   
33093 2018-08-02 13:00:00  0.0  0.053   0.079   0.055   0.057  39.14  34.80   

        T_20   T_50        Flag  
23935  33.75  30.77  2147483648  
23936  33.75  30.77  2147483648  
33086  30.35  31.74         

We now see that duplicate are present and that finding the unique rows of the df with all the duplicates is the same as setting keep='first' in drop_duplicates. Therefore, something we can expect is, the number of dropped duplicates should equal the number of rows in the df with all dups MINUS the number of rows in the df with unique duplicates.

In [7]:
def case_one_dup_drop_report(df, station, output_dir):
    """
    Expects df that was read in by read_data.py and the timestamps to be in the columns 
    
    Gets the duplicate dataframe and writes a text file with a report of the number of rows before and after dropping case one duplicates, for checking purpose.
    """
    
    # case one duplicates
    df_no_case_one_dups, df_showing_case_one_dups = treat_case_one_dups(df)

    # generate report content IF data was dropped
    if df_no_case_one_dups.shape[0] != df.shape[0]:

        print(f"Case one duplicates found for station {station}. Generating report...")

        # save duplicate df to csv in output_dir
        duplicate_output_file = f"{output_dir}/{station}.csv"
        df_showing_case_one_dups.to_csv(duplicate_output_file, index=False)
        
    return df_no_case_one_dups # always return the df with case one duplicates dropped, even if no duplicates were found

In [8]:
def case_two_dup_drop_report(df, station, output_dir, ignore_col='Flag'):
    """
    Expects df to be the dataframe after dropping case one duplicates, keeping the first occurrence, with the timestamps in the columns.
    
    Ignores the specified column when determining duplicates.
    
    Gets the duplicate dataframe and writes a text file with a report of the number of rows before and after dropping case two duplicates, for checking purpose.
    """

    df_no_case_two_dups, df_showing_case_two_dups = treat_case_two_dups(df, ignore_col=ignore_col)

    if df_showing_case_two_dups.empty: # if empty there is no flag then case two is impossible
        return df_no_case_two_dups # just the input df in this case

    # generate report IF data was dropped
    if df.shape[0] != df_no_case_two_dups.shape[0]:
        
        print(f"Case two duplicates found for station {station}. Generating report...")

        # save duplicate df to csv in output_dir
        duplicate_output_file = f"{output_dir}/{station}.csv"
        df_showing_case_two_dups.to_csv(duplicate_output_file, index=False)
            
    return df_no_case_two_dups # always return the df with case two duplicates dropped, even if no duplicates were found
        

Everything checks out for CB01. Reference read me on what to expect and check

# Dropping dups across all stations and recording

In [9]:
case_one_dup_df_folder = r"C:\pyt\tx-soil-moisture\data-cleanup-Su26\duplicate_report\full_row_duplicate"
case_two_dup_df_folder = r"C:\pyt\tx-soil-moisture\data-cleanup-Su26\duplicate_report\full_dup_but_flag"

# check for directory existence and create if not exist
if not os.path.exists(case_one_dup_df_folder):
    os.makedirs(case_one_dup_df_folder)

if not os.path.exists(case_two_dup_df_folder):
    os.makedirs(case_two_dup_df_folder)

data_dict = {}
for station, df in raw_data_dict.items():

    print(f"\033[32m\nProcessing {station}...\033[0m") # just prints it in green

    # reset index to ensure the index to consider duplicate timestamps
    timestamp_col_name = df.index.name
    df = df.reset_index() # CRUCIAL 
    
    df = case_one_dup_drop_report(df, station, output_dir=case_one_dup_df_folder)

    df = case_two_dup_drop_report(df, station, output_dir=case_two_dup_df_folder, ignore_col='Flag')

    # set the index back to the timestamp column
    df = df.set_index(timestamp_col_name)

    data_dict[station] = df



Processing CB01...


Case one duplicates found for station CB01. Generating report...
Case two duplicates found for station CB01. Generating report...

Processing CB01_met...
Case one duplicates found for station CB01_met. Generating report...
Column 'Flag' not found in the dataframe. No Case two duplicates will be dropped.

Processing CB04...
Case one duplicates found for station CB04. Generating report...

Processing CB04_met...
Case one duplicates found for station CB04_met. Generating report...
Column 'Flag' not found in the dataframe. No Case two duplicates will be dropped.

Processing CB06...
Case one duplicates found for station CB06. Generating report...
Case two duplicates found for station CB06. Generating report...

Processing CB06_met...
Case one duplicates found for station CB06_met. Generating report...
Column 'Flag' not found in the dataframe. No Case two duplicates will be dropped.

Processing CB07...
Case one duplicates found for station CB07. Generating report...

Processing CB09...
Case 

# Case Three duplicate

With case one and two duplicates handled, we can now handle the third case: Same timestamp but different measurements.

First we will find all occurrences of this case, then categorize them into three groups: on the day daylight savings time (DST) ends, exactly one day after  DST ends, and any other time.

In [10]:
def first_sunday_of_november(year):
    """ Returns the date of the first Sunday of November for a given year."""
    first = pd.Timestamp(year, 11, 1)
    days_to_sun = (6 - first.weekday())
    return first + pd.Timedelta(days=days_to_sun)

def is_on_dst_end(timestamp):
    # returns if the timestamps is on the day dst ends, at one am.
    dst_end_date = first_sunday_of_november(timestamp.year)
    return timestamp.date() == dst_end_date.date() and timestamp.hour == 1

def is_day_after_dst_end(timestamp):
    # returns if the timestamps is on the day AFTER dst ends, at one am.
    dst_end_date = first_sunday_of_november(timestamp.year)
    return timestamp.date() == (dst_end_date + pd.Timedelta(days=1)).date() and timestamp.hour == 1


In [11]:
def find_case_three_dups(df):
    """
    Finds case three duplicates in the dataframe. Expects the index to be the timestamps
    """
    
    # find duplicate timestamps, keeping all occurrences
    duplicate_timestamp_rows = df[df.index.duplicated(keep=False)]

    # subset data to a df for each category of duplicate timestamps: those that are on the DST end date, those that are the day after the DST end date, and those that are neither
    dst_end_dups = duplicate_timestamp_rows[duplicate_timestamp_rows.index.map(is_on_dst_end)]
    day_after_dst_end_dups = duplicate_timestamp_rows[duplicate_timestamp_rows.index.map(is_day_after_dst_end)]
    other_dups = duplicate_timestamp_rows[~duplicate_timestamp_rows.index.map(lambda ts: is_on_dst_end(ts) or is_day_after_dst_end(ts)).astype(bool)]

    # drop all case three dups and keep only the first occurrence, regardless of category
    df_no_case_three_dups = df[~df.index.duplicated(keep='first')]

    return dst_end_dups, day_after_dst_end_dups, other_dups, df_no_case_three_dups

In [13]:
any_other_time_path = r"C:\pyt\tx-soil-moisture\data-cleanup-Su26\duplicate_report\only_duplicate_timestamps\any_other_time"
day_after_dst_end_path = r"C:\pyt\tx-soil-moisture\data-cleanup-Su26\duplicate_report\only_duplicate_timestamps\day_after_dst_ends"
when_dst_ends_path = r"C:\pyt\tx-soil-moisture\data-cleanup-Su26\duplicate_report\only_duplicate_timestamps\when_dst_ends"

# check for directory existence and create if it does not exist
if not os.path.exists(any_other_time_path):
    os.makedirs(any_other_time_path)
if not os.path.exists(day_after_dst_end_path):
    os.makedirs(day_after_dst_end_path)
if not os.path.exists(when_dst_ends_path):
    os.makedirs(when_dst_ends_path)

# for each station, find and catagorize the case three dups, the download the duplicates in the appropriate folder.
for station, df in data_dict.items():
    print(f"\033[32m\nFinding case three duplicates for station {station}...\033[0m") # just prints it in green
    dst_end_dups, day_after_dst_end_dups, other_dups, no_dups = find_case_three_dups(df)
    
    if not dst_end_dups.empty:
        print(f"Case three duplicates found for station {station} ON THE DST END DATE:")
        # download the duplicate timestamps with measurements for those timestamps to a csv in when_dst_ends_path
        dst_end_dups.to_csv(f"{when_dst_ends_path}/{station}.csv")

    if not day_after_dst_end_dups.empty:
        print(f"Case three duplicates found for station {station} that are the day after the DST end date")
        # download the duplicate timestamps with measurements for those timestamps to a csv in day_after_dst_end_path
        day_after_dst_end_dups.to_csv(f"{day_after_dst_end_path}/{station}.csv")

    if not other_dups.empty:
        print(f"Case three duplicates found for station {station} that are not on the DST end date OR the day after:")
        # download the duplicate timestamps with measurements for those timestamps to a csv in any_other_time_path
        other_dups.to_csv(f"{any_other_time_path}/{station}.csv")


Finding case three duplicates for station CB01...
Case three duplicates found for station CB01 that are the day after the DST end date

Finding case three duplicates for station CB01_met...

Finding case three duplicates for station CB04...
Case three duplicates found for station CB04 that are the day after the DST end date

Finding case three duplicates for station CB04_met...
Case three duplicates found for station CB04_met that are not on the DST end date OR the day after:

Finding case three duplicates for station CB06...
Case three duplicates found for station CB06 that are the day after the DST end date

Finding case three duplicates for station CB06_met...

Finding case three duplicates for station CB07...
Case three duplicates found for station CB07 that are not on the DST end date OR the day after:

Finding case three duplicates for station CB09...
Case three duplicates found for station CB09 that are the day after the DST end date

Finding case three duplicates for station C